# Datathon 2026 - Core LightGBM Forecasting Notebook

This notebook is designed to run on Kaggle. It can use prebuilt `model_core_*` Parquet files if they are attached, or rebuild the leakage-safe core dataset from raw `sales.csv` and `sample_submission.csv`.

Workflow:

- Build/load core train, valid, test, and future datasets
- Train separate models for `Revenue` and `COGS`
- Evaluate on the time-based validation and benchmark splits
- Save feature importance and SHAP plots when available
- Refit on full historical data and recursively forecast `2023-01-01 -> 2024-07-01`
- Export `outputs/submission.csv`

Fixed split:

- Train: `2012-07-04 -> 2017-12-31`
- Valid: `2018-01-01 -> 2019-12-31`
- Test benchmark: `2020-01-01 -> 2022-12-31`
- Future/submission: dates from `sample_submission.csv`

In [ ]:
# -*- coding: utf-8 -*-
from pathlib import Path
import json
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

try:
    import lightgbm as lgb
    USE_LGBM = True
except Exception as exc:
    print("LightGBM unavailable; falling back to sklearn HistGradientBoostingRegressor:", exc)
    from sklearn.ensemble import HistGradientBoostingRegressor
    USE_LGBM = False

try:
    import shap
    HAS_SHAP = True
except Exception as exc:
    print("SHAP unavailable; feature importance CSVs will still be saved:", exc)
    HAS_SHAP = False

try:
    import joblib
except Exception:
    joblib = None

KAGGLE_WORKING = Path("/kaggle/working")
ROOT = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd()
OUTPUT_DIR = ROOT / "outputs"
MODEL_DIR = ROOT / "models"
DATASET_DIR = ROOT / "dataset"
for p in [OUTPUT_DIR, MODEL_DIR, DATASET_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("USE_LGBM:", USE_LGBM)
print("HAS_SHAP:", HAS_SHAP)

## 1. Locate Input Data

The notebook first looks for processed Parquet files. If they are not available, it searches for raw `sales.csv` and `sample_submission.csv` under `/kaggle/input` and rebuilds the core features.

In [ ]:
def find_dir_with(files, search_roots):
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        candidates = list(root.rglob(files[0])) if root.is_dir() else []
        for first_file in candidates:
            directory = first_file.parent
            if all((directory / name).exists() for name in files):
                return directory
    return None

SEARCH_ROOTS = [Path("/kaggle/input"), Path.cwd(), Path.cwd() / "data", Path.cwd() / "dataset"]
PROCESSED_DIR = find_dir_with(["model_core_train.parquet", "model_core_valid.parquet", "model_core_test.parquet", "model_core_future.parquet"], SEARCH_ROOTS)
RAW_DATA_DIR = find_dir_with(["sales.csv", "sample_submission.csv"], SEARCH_ROOTS)

print("PROCESSED_DIR:", PROCESSED_DIR)
print("RAW_DATA_DIR:", RAW_DATA_DIR)

if RAW_DATA_DIR is None and PROCESSED_DIR is None:
    raise FileNotFoundError("Could not find either processed Parquet files or raw sales.csv/sample_submission.csv under /kaggle/input or current directory.")

## 2. Leakage-Safe Feature Builder

Rule: for a row dated `t`, every non-calendar feature must use information available no later than `t-1`. Calendar and deterministic holiday features for date `t` are allowed because they are known in advance.

In [ ]:
DATE_COL = "Date"
TARGETS = ["Revenue", "COGS"]

TRAIN_END = pd.Timestamp("2017-12-31")
VALID_START = pd.Timestamp("2018-01-01")
VALID_END = pd.Timestamp("2019-12-31")
TEST_START = pd.Timestamp("2020-01-01")
TEST_END = pd.Timestamp("2022-12-31")

LAGS = [1, 2, 3, 7, 14, 21, 28, 30, 56, 90, 180, 364, 365]
ROLL_WINDOWS = [3, 7, 14, 28, 30, 56, 90, 180, 365]
EWMA_SPANS = [7, 14, 30, 90]

TET_DATES = {
    2013: "2013-02-10", 2014: "2014-01-31", 2015: "2015-02-19",
    2016: "2016-02-08", 2017: "2017-01-28", 2018: "2018-02-16",
    2019: "2019-02-05", 2020: "2020-01-25", 2021: "2021-02-12",
    2022: "2022-02-01", 2023: "2023-01-22", 2024: "2024-02-10",
}
FIXED_HOLIDAYS = {(1, 1), (4, 30), (5, 1), (9, 2)}
SALE_DAYS = {(11, 11), (12, 12)}


def add_calendar_features(df, base_year=None):
    out = df.copy()
    date = out[DATE_COL]
    iso = date.dt.isocalendar()
    if base_year is None:
        base_year = int(date.dt.year.min())

    out["year"] = date.dt.year
    out["year_index"] = out["year"] - base_year
    out["quarter"] = date.dt.quarter
    out["month"] = date.dt.month
    out["weekofyear"] = iso.week.astype("int16")
    out["dayofyear"] = date.dt.dayofyear
    out["dayofmonth"] = date.dt.day
    out["dayofweek"] = date.dt.dayofweek
    out["is_weekend"] = out["dayofweek"].isin([5, 6]).astype("int8")
    out["is_month_start"] = date.dt.is_month_start.astype("int8")
    out["is_month_end"] = date.dt.is_month_end.astype("int8")
    out["is_quarter_start"] = date.dt.is_quarter_start.astype("int8")
    out["is_quarter_end"] = date.dt.is_quarter_end.astype("int8")
    out["is_year_start"] = date.dt.is_year_start.astype("int8")
    out["is_year_end"] = date.dt.is_year_end.astype("int8")

    for period, col in [(7, "dow"), (12, "month"), (365.25, "year")]:
        source = out["dayofweek"] if col == "dow" else out["month"] if col == "month" else out["dayofyear"]
        out[f"{col}_sin"] = np.sin(2 * np.pi * source / period)
        out[f"{col}_cos"] = np.cos(2 * np.pi * source / period)
    return out


def add_holiday_features(df):
    out = df.copy()
    dates = out[DATE_COL]
    tet_dates = pd.to_datetime(list(TET_DATES.values()))

    def tet_distance(date):
        deltas = (tet_dates - date).days
        future = deltas[deltas >= 0]
        past = deltas[deltas < 0]
        days_to = int(future.min()) if len(future) else 999
        days_after = int(-past.max()) if len(past) else 999
        return days_to, days_after

    proximity = dates.apply(tet_distance)
    out["days_to_tet"] = [item[0] for item in proximity]
    out["days_after_tet"] = [item[1] for item in proximity]
    out["tet_proximity"] = np.minimum(out["days_to_tet"], out["days_after_tet"])
    out["is_tet_week"] = ((out["days_to_tet"].le(7)) | (out["days_after_tet"].le(7))).astype("int8")
    out["is_pre_tet_2w"] = out["days_to_tet"].between(1, 14, inclusive="both").astype("int8")
    out["is_pre_tet_month"] = out["days_to_tet"].between(1, 30, inclusive="both").astype("int8")
    out["is_fixed_holiday"] = dates.apply(lambda d: int((d.month, d.day) in FIXED_HOLIDAYS)).astype("int8")
    out["is_sale_day"] = dates.apply(lambda d: int((d.month, d.day) in SALE_DAYS)).astype("int8")
    out["is_1111"] = ((out["month"] == 11) & (out["dayofmonth"] == 11)).astype("int8")
    out["is_1212"] = ((out["month"] == 12) & (out["dayofmonth"] == 12)).astype("int8")
    out["is_black_friday"] = dates.apply(lambda d: int(d.month == 11 and d.weekday() == 4 and (d.day - 1) // 7 == 3)).astype("int8")
    return out


def add_target_lag_features(df):
    out = df.copy().sort_values(DATE_COL).reset_index(drop=True)
    for target in TARGETS:
        shifted = out[target].shift(1)
        for lag in LAGS:
            out[f"{target}_lag_{lag}"] = out[target].shift(lag)
        for window in ROLL_WINDOWS:
            roll = shifted.rolling(window=window, min_periods=max(2, min(7, window)))
            out[f"{target}_roll{window}_mean"] = roll.mean()
            out[f"{target}_roll{window}_std"] = roll.std()
            out[f"{target}_roll{window}_min"] = roll.min()
            out[f"{target}_roll{window}_max"] = roll.max()
            out[f"{target}_roll{window}_median"] = roll.median()
        for span in EWMA_SPANS:
            out[f"{target}_ewm{span}_mean"] = shifted.ewm(span=span, adjust=False, min_periods=2).mean()
        out[f"{target}_lag1_vs_roll7"] = out[f"{target}_lag_1"] / out[f"{target}_roll7_mean"]
        out[f"{target}_roll7_vs_roll30"] = out[f"{target}_roll7_mean"] / out[f"{target}_roll30_mean"]
        out[f"{target}_roll30_vs_roll90"] = out[f"{target}_roll30_mean"] / out[f"{target}_roll90_mean"]
        out[f"{target}_roll90_vs_roll365"] = out[f"{target}_roll90_mean"] / out[f"{target}_roll365_mean"]

    out["gross_margin_lag_1"] = out["Revenue_lag_1"] - out["COGS_lag_1"]
    out["gross_margin_rate_lag_1"] = out["gross_margin_lag_1"] / out["Revenue_lag_1"]
    out["revenue_cogs_ratio_lag_1"] = out["Revenue_lag_1"] / out["COGS_lag_1"]
    return out.replace([np.inf, -np.inf], np.nan)


def add_prior_seasonal_profile(history, future):
    hist = history.copy()
    fut = future.copy()
    for frame in (hist, fut):
        frame["season_month"] = frame[DATE_COL].dt.month
        frame["season_day"] = frame[DATE_COL].dt.day

    future_global_stats = hist[TARGETS].agg(["mean", "std"]).to_dict()
    for target in TARGETS:
        group = hist.groupby(["season_month", "season_day"], group_keys=False)[target]
        hist[f"{target}_md_prior_mean"] = group.transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
        hist[f"{target}_md_prior_std"] = group.transform(lambda s: s.shift(1).expanding(min_periods=2).std())
        hist[f"{target}_md_prior_mean"] = hist[f"{target}_md_prior_mean"].fillna(hist[target].shift(1).expanding(min_periods=1).mean())
        hist[f"{target}_md_prior_std"] = hist[f"{target}_md_prior_std"].fillna(hist[target].shift(1).expanding(min_periods=2).std())

        md_stats = (
            hist.groupby(["season_month", "season_day"])[target]
            .agg(["mean", "std"])
            .rename(columns={"mean": f"{target}_md_prior_mean", "std": f"{target}_md_prior_std"})
            .reset_index()
        )
        fut = fut.merge(md_stats, on=["season_month", "season_day"], how="left")
        fut[f"{target}_md_prior_mean"] = fut[f"{target}_md_prior_mean"].fillna(future_global_stats[target]["mean"])
        fut[f"{target}_md_prior_std"] = fut[f"{target}_md_prior_std"].fillna(future_global_stats[target]["std"])

    return hist.drop(columns=["season_month", "season_day"]), fut.drop(columns=["season_month", "season_day"])

In [ ]:
def build_core_from_raw(raw_data_dir):
    raw_data_dir = Path(raw_data_dir)
    sales = pd.read_csv(raw_data_dir / "sales.csv", parse_dates=[DATE_COL]).sort_values(DATE_COL).reset_index(drop=True)
    future = pd.read_csv(raw_data_dir / "sample_submission.csv", parse_dates=[DATE_COL])[[DATE_COL]].sort_values(DATE_COL).reset_index(drop=True)
    for target in TARGETS:
        future[target] = np.nan

    base_year = int(sales[DATE_COL].dt.year.min())
    history = add_calendar_features(sales, base_year=base_year)
    future = add_calendar_features(future, base_year=base_year)
    history = add_holiday_features(history)
    future = add_holiday_features(future)
    history = add_target_lag_features(history)
    history, future = add_prior_seasonal_profile(history, future)

    for col in history.columns:
        if col not in future.columns:
            future[col] = np.nan
    future = future[history.columns]
    return history, future, sales


def split_core(history):
    return {
        "train": history[history[DATE_COL].le(TRAIN_END)].copy(),
        "valid": history[history[DATE_COL].between(VALID_START, VALID_END, inclusive="both")].copy(),
        "test": history[history[DATE_COL].between(TEST_START, TEST_END, inclusive="both")].copy(),
        "history": history.copy(),
    }


def validate_splits(train, valid, test):
    assert train[DATE_COL].max() <= TRAIN_END
    assert valid[DATE_COL].min() >= VALID_START and valid[DATE_COL].max() <= VALID_END
    assert test[DATE_COL].min() >= TEST_START and test[DATE_COL].max() <= TEST_END
    assert set(train[DATE_COL]).isdisjoint(set(valid[DATE_COL]))
    assert set(train[DATE_COL]).isdisjoint(set(test[DATE_COL]))
    assert set(valid[DATE_COL]).isdisjoint(set(test[DATE_COL]))
    print(f"train: {len(train):,} rows | {train.Date.min().date()} -> {train.Date.max().date()}")
    print(f"valid: {len(valid):,} rows | {valid.Date.min().date()} -> {valid.Date.max().date()}")
    print(f"test : {len(test):,} rows | {test.Date.min().date()} -> {test.Date.max().date()}")
    print("Split assertions passed.")


if PROCESSED_DIR is not None:
    train = pd.read_parquet(PROCESSED_DIR / "model_core_train.parquet")
    valid = pd.read_parquet(PROCESSED_DIR / "model_core_valid.parquet")
    test = pd.read_parquet(PROCESSED_DIR / "model_core_test.parquet")
    future_template = pd.read_parquet(PROCESSED_DIR / "model_core_future.parquet")
    history_features = pd.read_parquet(PROCESSED_DIR / "model_core_history.parquet") if (PROCESSED_DIR / "model_core_history.parquet").exists() else pd.concat([train, valid, test], ignore_index=True)
    if RAW_DATA_DIR is not None:
        raw_sales = pd.read_csv(Path(RAW_DATA_DIR) / "sales.csv", parse_dates=[DATE_COL]).sort_values(DATE_COL).reset_index(drop=True)
    else:
        raw_sales = history_features[[DATE_COL, "Revenue", "COGS"]].dropna().copy()
else:
    history_features, future_template, raw_sales = build_core_from_raw(RAW_DATA_DIR)
    splits = split_core(history_features)
    train, valid, test = splits["train"], splits["valid"], splits["test"]
    for name, frame in {"model_core_history": history_features, "model_core_train": train, "model_core_valid": valid, "model_core_test": test, "model_core_future": future_template}.items():
        frame.to_parquet(DATASET_DIR / f"{name}.parquet", index=False)

validate_splits(train, valid, test)
print("history_features:", history_features.shape)
print("future_template :", future_template.shape)

## 3. Train LightGBM Models

Targets are transformed with `log1p`; predictions are inverted with `expm1` and clipped to non-negative values.

In [ ]:
FEATURE_COLS = [c for c in train.columns if c not in [DATE_COL, "Revenue", "COGS"]]
print("Feature count:", len(FEATURE_COLS))

LGB_PARAMS = {
    "objective": "regression",
    "metric": ["mae", "rmse"],
    "num_leaves": 63,
    "learning_rate": 0.03,
    "n_estimators": 3000,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.75,
    "bagging_freq": 5,
    "min_child_samples": 15,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "verbose": -1,
    "seed": SEED,
    "random_state": SEED,
    "deterministic": True,
}


def make_estimator(n_estimators=None):
    if USE_LGBM:
        params = LGB_PARAMS.copy()
        if n_estimators is not None:
            params["n_estimators"] = int(n_estimators)
        return lgb.LGBMRegressor(**params)
    return HistGradientBoostingRegressor(
        max_iter=int(n_estimators or 300),
        learning_rate=0.03,
        max_leaf_nodes=63,
        l2_regularization=0.1,
        random_state=SEED,
    )


def train_model(train_df, valid_df, target):
    X_train = train_df[FEATURE_COLS]
    y_train = np.log1p(train_df[target])
    X_valid = valid_df[FEATURE_COLS]
    y_valid = np.log1p(valid_df[target])

    model = make_estimator()
    if USE_LGBM:
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(200)],
        )
    else:
        model.fit(X_train, y_train)
    return model


def predict_nonnegative(model, X):
    return np.expm1(model.predict(X)).clip(min=0)


def evaluate_predictions(y_true, preds):
    return {
        "MAE": float(mean_absolute_error(y_true, preds)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, preds))),
        "R2": float(r2_score(y_true, preds)),
    }


model_revenue = train_model(train, valid, "Revenue")
model_cogs = train_model(train, valid, "COGS")

if USE_LGBM:
    best_iter_revenue = int(getattr(model_revenue, "best_iteration_", 0) or LGB_PARAMS["n_estimators"])
    best_iter_cogs = int(getattr(model_cogs, "best_iteration_", 0) or LGB_PARAMS["n_estimators"])
else:
    best_iter_revenue = 300
    best_iter_cogs = 300
print("best_iter_revenue:", best_iter_revenue)
print("best_iter_cogs   :", best_iter_cogs)

## 4. Validation and Test Benchmark

Use the validation split for tuning. Use the test split as the final benchmark after the model configuration is fixed.

In [ ]:
def evaluate_split(model, df, target, split_name):
    preds = predict_nonnegative(model, df[FEATURE_COLS])
    metrics = evaluate_predictions(df[target].values, preds)
    print(f"[{split_name} | {target}] MAE={metrics['MAE']:.2f} RMSE={metrics['RMSE']:.2f} R2={metrics['R2']:.4f}")
    return metrics, preds

metrics = []
for split_name, df in [("valid", valid), ("test", test)]:
    for target, model in [("Revenue", model_revenue), ("COGS", model_cogs)]:
        m, preds = evaluate_split(model, df, target, split_name)
        metrics.append({"split": split_name, "target": target, **m})

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(OUTPUT_DIR / "core_lgb_metrics.csv", index=False)
metrics_df

## 5. Feature Importance and SHAP

SHAP runs only when the package is available. Feature-importance CSV files are always saved when the estimator exposes importances.

In [ ]:
def save_feature_importance(model, label):
    if not hasattr(model, "feature_importances_"):
        return None
    imp = pd.DataFrame({
        "feature": FEATURE_COLS,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)
    path = OUTPUT_DIR / f"feature_importance_{label}.csv"
    imp.to_csv(path, index=False)
    print("Saved", path)
    return imp

imp_revenue = save_feature_importance(model_revenue, "revenue")
imp_cogs = save_feature_importance(model_cogs, "cogs")
if imp_revenue is not None:
    display(imp_revenue.head(20))

In [ ]:
def run_shap(model, df, label, max_rows=600):
    if not (USE_LGBM and HAS_SHAP):
        print(f"Skipping SHAP for {label}.")
        return None
    X = df[FEATURE_COLS].copy()
    if len(X) > max_rows:
        X = X.sample(max_rows, random_state=SEED).sort_index()

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    shap.summary_plot(shap_values, X, max_display=25, show=False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"shap_{label}_summary.png", dpi=160, bbox_inches="tight")
    plt.close()

    shap.summary_plot(shap_values, X, plot_type="bar", max_display=25, show=False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"shap_{label}_importance.png", dpi=160, bbox_inches="tight")
    plt.close()

    target_date = pd.Timestamp("2022-02-01")
    candidate_positions = np.where(df.loc[X.index, DATE_COL].values == np.datetime64(target_date))[0]
    if len(candidate_positions):
        pos = int(candidate_positions[0])
        exp = shap.Explanation(
            values=shap_values[pos],
            base_values=explainer.expected_value,
            data=X.iloc[pos],
            feature_names=FEATURE_COLS,
        )
        shap.plots.waterfall(exp, show=False, max_display=20)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"shap_{label}_tet_waterfall.png", dpi=160, bbox_inches="tight")
        plt.close()

    print(f"Saved SHAP plots for {label}.")
    return shap_values

_ = run_shap(model_revenue, test, "revenue")
_ = run_shap(model_cogs, test, "cogs")

## 6. Refit on Full History and Recursive Future Forecast

After selecting the configuration with validation and benchmarking on test, refit the final models on all historical rows through `2022-12-31`, then recursively forecast the future dates.

In [ ]:
def fit_final_model(history_df, target, best_iter):
    model = make_estimator(n_estimators=best_iter)
    X = history_df[FEATURE_COLS]
    y = np.log1p(history_df[target])
    model.fit(X, y)
    return model

final_revenue_model = fit_final_model(history_features, "Revenue", best_iter_revenue)
final_cogs_model = fit_final_model(history_features, "COGS", best_iter_cogs)

if joblib is not None:
    joblib.dump(final_revenue_model, MODEL_DIR / "final_revenue_model.joblib")
    joblib.dump(final_cogs_model, MODEL_DIR / "final_cogs_model.joblib")
    print("Saved final models to", MODEL_DIR)

In [ ]:
def make_seasonal_lookup(real_history):
    temp = real_history[[DATE_COL, "Revenue", "COGS"]].copy()
    temp["season_month"] = temp[DATE_COL].dt.month
    temp["season_day"] = temp[DATE_COL].dt.day
    lookup = {}
    global_stats = temp[TARGETS].agg(["mean", "std"]).to_dict()
    for target in TARGETS:
        stats = temp.groupby(["season_month", "season_day"])[target].agg(["mean", "std"])
        lookup[target] = {"stats": stats, "global": global_stats[target]}
    return lookup


def add_fixed_seasonal_priors(row, seasonal_lookup):
    month = int(row["month"].iloc[0])
    day = int(row["dayofmonth"].iloc[0])
    for target in TARGETS:
        stats = seasonal_lookup[target]["stats"]
        global_stats = seasonal_lookup[target]["global"]
        if (month, day) in stats.index:
            mean_value = stats.loc[(month, day), "mean"]
            std_value = stats.loc[(month, day), "std"]
        else:
            mean_value = global_stats["mean"]
            std_value = global_stats["std"]
        row[f"{target}_md_prior_mean"] = mean_value if pd.notna(mean_value) else global_stats["mean"]
        row[f"{target}_md_prior_std"] = std_value if pd.notna(std_value) else global_stats["std"]
    return row


def make_recursive_feature_row(buffer, date, feature_cols, seasonal_lookup, base_year):
    tmp = pd.concat([
        buffer[[DATE_COL, "Revenue", "COGS"]],
        pd.DataFrame({DATE_COL: [date], "Revenue": [np.nan], "COGS": [np.nan]})
    ], ignore_index=True).sort_values(DATE_COL).reset_index(drop=True)

    tmp = add_calendar_features(tmp, base_year=base_year)
    tmp = add_holiday_features(tmp)
    tmp = add_target_lag_features(tmp)
    row = tmp[tmp[DATE_COL].eq(date)].tail(1).copy()
    row = add_fixed_seasonal_priors(row, seasonal_lookup)

    for col in feature_cols:
        if col not in row.columns:
            row[col] = np.nan
    return row[feature_cols]


def recursive_forecast(revenue_model, cogs_model, real_history, future_dates, feature_cols, margin_floor=0.05):
    base_year = int(real_history[DATE_COL].dt.year.min())
    seasonal_lookup = make_seasonal_lookup(real_history)
    buffer = real_history[[DATE_COL, "Revenue", "COGS"]].copy().sort_values(DATE_COL).reset_index(drop=True)
    predictions = []

    for date in pd.to_datetime(future_dates):
        row = make_recursive_feature_row(buffer, date, feature_cols, seasonal_lookup, base_year)
        revenue_pred = float(predict_nonnegative(revenue_model, row)[0])
        cogs_pred = float(predict_nonnegative(cogs_model, row)[0])
        cogs_pred = min(max(cogs_pred, 0.0), revenue_pred * (1 - margin_floor))
        revenue_pred = max(revenue_pred, 0.0)

        predictions.append({DATE_COL: date, "Revenue": revenue_pred, "COGS": cogs_pred})
        buffer = pd.concat([buffer, pd.DataFrame([{DATE_COL: date, "Revenue": revenue_pred, "COGS": cogs_pred}])], ignore_index=True)

    return pd.DataFrame(predictions)

future_dates = future_template[DATE_COL]
future_preds = recursive_forecast(final_revenue_model, final_cogs_model, raw_sales, future_dates, FEATURE_COLS)
future_preds.head()

## 7. Submission

In [ ]:
submission = future_preds.copy()
submission[DATE_COL] = submission[DATE_COL].dt.strftime("%Y-%m-%d")
submission["Revenue"] = submission["Revenue"].round(2)
submission["COGS"] = submission["COGS"].round(2)

assert len(submission) == len(future_template), "Row count mismatch"
assert submission["Revenue"].isna().sum() == 0, "NaN in Revenue"
assert submission["COGS"].isna().sum() == 0, "NaN in COGS"
assert (submission["Revenue"] >= 0).all(), "Negative Revenue"
assert (submission["COGS"] >= 0).all(), "Negative COGS"
assert (submission["COGS"] < submission["Revenue"]).mean() > 0.99, "COGS >= Revenue too often"

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)
print("Saved", submission_path)
submission.head(10)

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(raw_sales[DATE_COL], raw_sales["Revenue"], label="Historical Revenue", linewidth=0.8)
plt.plot(pd.to_datetime(submission[DATE_COL]), submission["Revenue"], label="Forecast Revenue", linewidth=0.8)
plt.title("Revenue Forecast")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "forecast_revenue.png", dpi=160, bbox_inches="tight")
plt.show()